**[Source]** Donghwan Project (`맞춤법교정_정동환.ipynb`의 모델 구조·후보 설명 부분) + Jisoo Project 6번(구조 확인 결과 재사용)
**[Status]** ADAPTED
**[Role]** Encoder–Decoder 구조 설명, 모델 후보 정리, 지수 데이터 구조와의 호환 규칙(Compatibility Change) 정리
**[Modification]** 동환의 전처리·split·길이 이상치 제거는 가져오지 않았다(지수 FIXED 데이터가 대체). 모델 후보는 동환 노트북에 있다는 이유만으로 고정하지 않고, 지수의 실제 실행 결과를 우선 확인했다.
**이 노트북은 GPU 없이 실행된다(파일 확인·표 작성). 모델을 새로 학습하지 않는다.**

# 06. Transformer Encoder–Decoder 구조와 모델 후보

## 1. 과제 정의 — 분류가 아니라 생성
입력은 **오류가 있는 한 문장**(`input`), 출력은 **교정된 한 문장 전체**(`target`)다. 오류 유무를 고르는 분류가 아니라 문장을 처음부터 다시 생성하는 Seq2Seq 과제이므로 Encoder–Decoder 구조가 적합하다(입력을 이해하는 부분과 출력을 만드는 부분이 분리됨).

## 2. 구조와 학습/추론의 차이 (동환 설명을 지수 6번 확인 결과와 맞춰 정리)
| 구성 | 하는 일 |
|---|---|
| Encoder | 입력 문장 전체를 양방향 Self-Attention으로 읽어 각 토큰의 문맥 표현을 만든다. |
| Decoder Masked Self-Attention | 지금까지 만든 출력 토큰만 보게 한다(causal mask로 미래 토큰을 가림). |
| Cross-Attention | Decoder가 다음 토큰을 정할 때 Encoder 출력(원문 문맥)을 참조한다. |
| 학습(Teacher Forcing) | Decoder 입력 = 정답을 한 칸 오른쪽으로 민 시퀀스(shift-right). 정답을 알려 주고 모든 위치의 “다음 토큰”을 병렬로 맞히게 하며, loss는 토큰별 Cross-Entropy. padding 위치는 label을 `-100`으로 두어 loss에서 제외한다. |
| 추론(Generation) | 정답이 없으므로 Decoder 입력 = 모델이 이미 생성한 토큰. 한 토큰씩 만들고 EOS가 나오면 멈춘다(Greedy 또는 Beam Search). |

학습과 추론에서 Decoder 입력이 다르기 때문에, **학습 loss가 낮아도 생성 결과는 별도로 평가**해야 한다(Exposure bias). Attention mask(padding 가림)와 causal mask(미래 가림)는 서로 다른 것이다.

## 3. 후보 구조
KoBART는 BART 계열, pko-T5와 ET5는 T5 계열 Encoder–Decoder다(논문 기준 일반 설명: BART=noising 복원 사전학습·학습형 위치 임베딩, T5=text-to-text·span corruption·상대 위치 편향). 아래 세부 설정은 **파일로 확인한 값과 확인하지 못한 값을 구분해서** 적는다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 지수 6번(구조 분석)·5번(tokenizer 분석)이 이미 확인해 둔 실제 결과를 읽는다 — 다시 계산하지 않는다
SC = json.loads((P.J_OUT / "model_structure" / "structure_config.json").read_text(encoding="utf-8"))
TK = json.loads((P.J_OUT / "tokenizer_analysis" / "tokenizer_final_config.json").read_text(encoding="utf-8"))
assert SC["dataset_version"] == MAN["dataset_version"] == TK["dataset_version"], "지수 구조/tokenizer 분석이 다른 데이터 버전으로 수행됨"
assert TK["dataset_sha256"]["train.jsonl"] == MAN["sha256"]["train.jsonl"], "tokenizer 분석에 쓴 Train과 현재 FIXED Train이 다름"
rows = []
for m in ("KoBART", "pko-T5"):
    o, f, c = SC["observed"][m], SC["final_structure_settings"][m], TK["candidates"][m]
    rows.append({"모델": m, "model_id": c["model_id"], "tokenizer": c["tokenizer_class"], "vocab": c["vocab_size"], "max_length(in/out)": f"{f['input max_length']}/{f['target max_length']}",
                 "Train 잘림%(in/out)": f"{100*c['train_trunc_pct']['input']:.3f}/{100*c['train_trunc_pct']['target']:.3f}", "target EOS": f["target EOS 정책"][:34], "decoder_start": f["decoder_start_token"],
                 "shift-right 자동==labels만 전달": o["loss_auto_equals_prepare"], "label pad": f["label padding"]})
print("[지수 5·6번에서 확인된 후보 구조/토큰 설정]"); print(pd.DataFrame(rows).set_index("모델").T.to_string())
print("\n지수 5번 결정 규칙:", TK["decision_rule"])
print("사용 라이브러리(지수 실행 기록): transformers", SC["transformers_version"], "| torch", SC["torch_version"])

[지수 5·6번에서 확인된 후보 구조/토큰 설정]
모델                                                                  KoBART                                             pko-T5
model_id                                              gogamza/kobart-base-v2                                  paust/pko-t5-base
tokenizer                                            PreTrainedTokenizerFast                                    T5TokenizerFast
vocab                                                                  30000                                              50258
max_length(in/out)                                                     56/56                                              72/72
Train 잘림%(in/out)                                              7.824/6.983                                        6.962/9.000
target EOS                       tokenizer가 EOS를 붙이지 않음: labels 끝에   tokenizer가 EOS를 자동으로 붙임: 직접 추가하지 않
decoder_start                                                       </s> (1)                            

In [3]:
# [셀 2] ET5(동환 후보) 사용 가능 여부 확인 — 파일이 없으면 있는 것처럼 쓰지 않는다
cands = [P.ET5_DIR] if P.ET5_DIR else []
cands += [P.JISOO_ROOT / "models" / "et5-base", P.ROOT / "models" / "et5-base", Path.cwd() / "models" / "et5-base"]
ET5_FOUND = next((c for c in cands if c and Path(c).exists() and (Path(c) / "config.json").exists()), None)
print("ET5 base 탐색 경로:", [str(c) for c in cands if c])
print("→ 발견:", ET5_FOUND)
ET5_CFG = None
if ET5_FOUND:
    ET5_CFG = json.loads((Path(ET5_FOUND) / "config.json").read_text(encoding="utf-8"))
    print({k: ET5_CFG.get(k) for k in ("model_type", "d_model", "num_layers", "num_decoder_layers", "vocab_size", "d_ff", "num_heads")})
else:
    print("ET5 base 가중치가 이 폴더 구조에서 발견되지 않았습니다. `config/paths.json`의 et5_model_dir에 경로를 지정하면 08번의 ET5 screening을 실행할 수 있습니다.")
    print("참고: 지수 폴더의 models/et5-typos-corrector 는 이름·용도로 보아 이미 맞춤법 교정용으로 학습된 모델로 보이며(내용 미확인), 사전학습만 된 공정한 후보가 아닐 수 있어 후보에서 제외.")
DH_REPORTED = {"KoBART": "Encoder 6층·Decoder 6층, 약 1.24억 파라미터, vocab 30,000", "ET5": "Encoder 12층·Decoder 12층, 약 3.24억 파라미터, vocab 45,100, 입력 앞에 '맞춤법 교정: ' 접두어"}
print("\n[동환 보고서에 기재된 값 — 이 환경에서 재확인하지 못함]"); 
for _k, _v in DH_REPORTED.items(): print(" ", _k, ":", _v)

ET5 base 탐색 경로: ['/sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/models/et5-base', '/sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/models/et5-base', '/sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/notebooks/models/et5-base']
→ 발견: None
ET5 base 가중치가 이 폴더 구조에서 발견되지 않았습니다. `config/paths.json`의 et5_model_dir에 경로를 지정하면 08번의 ET5 screening을 실행할 수 있습니다.
참고: 지수 폴더의 models/et5-typos-corrector 는 이름·용도로 보아 이미 맞춤법 교정용으로 학습된 모델로 보이며(내용 미확인), 사전학습만 된 공정한 후보가 아닐 수 있어 후보에서 제외.

[동환 보고서에 기재된 값 — 이 환경에서 재확인하지 못함]
  KoBART : Encoder 6층·Decoder 6층, 약 1.24억 파라미터, vocab 30,000
  ET5 : Encoder 12층·Decoder 12층, 약 3.24억 파라미터, vocab 45,100, 입력 앞에 '맞춤법 교정: ' 접두어


## 4. 직접 결정한 것과 사전학습 모델에 이미 들어 있는 것
| 사전학습 모델에 이미 포함(우리가 설계하지 않음) | 우리가 직접 결정한 것 |
|---|---|
| Encoder/Decoder 층 수, hidden 크기, attention head 수 | 어떤 사전학습 모델을 후보로 삼을지 / 선택 기준(Validation) |
| 사전학습된 embedding과 위치 표현 | tokenizer 확인 및 max_length(지수 5번: Train 잘림 ≤0.1% 기준) |
| tokenizer vocabulary | learning rate·batch·epoch·optimizer·scheduler(07번) |
| 사전학습 목적(BART: noising 복원, T5: span corruption) | 입력/정답 구성(input→target, 접두어, EOS 처리), 디코딩 설정(Greedy/Beam) |

라이브러리에서 모델을 불러왔다는 것만으로 “모델 설계”라고 주장하지 않는다. 이 프로젝트의 설계 판단은 오른쪽 열이다.

## 5. Compatibility Change (지수 데이터에 동환 코드를 맞춘 부분)
| 동환 원본 | 통합본 | 이유 |
|---|---|---|
| `original_form`/`form` → source, `corrected_form` → target | 지수 `input` → source 역할, `target` → target 역할(**adapter 함수에서만 매핑**, 지수 파일은 수정하지 않음) | 지수 데이터가 이미 정규화(NFC+strip)된 `input`/`target`을 제공 |
| JSON→SQLite 적재, 결측·중복 제거, 자체 document 해시 분할(80/10/10) | **제거** | 지수 FIXED 전처리·분할이 대체(05번에서 검증). 다시 나누면 지수 분할과 충돌 |
| 길이 상위 ~1% 문장을 Train/Validation/Test에서 모두 제거 | **제거**(Test는 전체 사용). 길이 처리는 지수 5번 기준(truncation) | 평가 세트에서 어려운 긴 문장을 빼면 점수가 부풀 수 있음 |
| Train 20,000 / Test 2,000 표본 | Train 전체 또는 지수 고정 subset(100,000/5,000), Validation·Test 전체 | 표본 평가의 변동성 제거, 지수 subset 결과와 직접 비교 |
| 평가 문자열 NFKC 정규화 | **유지**(지수 원문 기준 지표와 함께 보고) | 지수 5번이 pko-T5 tokenizer 복원 시 5,000행 중 24.12%가 NFKC 후에야 원문과 같아진다고 이미 측정함 |
| ET5 입력 접두어 `맞춤법 교정: ` | ET5에만 유지(모델별 입력 구성) | T5 계열의 text-to-text 관례. pko-T5는 지수 6번 설정(접두어 없음) 유지 |

In [4]:
# [셀 3] 후보 정리 저장 (다음 노트북들이 읽는다)
CAND = {
 "dataset_version": MAN["dataset_version"],
 "KoBART": {"model_id": "gogamza/kobart-base-v2", "family": "BART", "max_length": [56, 56], "input_prefix": "", "target_eos": "수동 추가 1개(지수 6번)", "source": "지수 5·6번 확인 결과"},
 "pko-T5": {"model_id": "paust/pko-t5-base", "family": "T5", "max_length": [72, 72], "input_prefix": "", "target_eos": "tokenizer 자동(중복 금지)", "source": "지수 5·6번 확인 결과"},
 "ET5": {"model_id": str(ET5_FOUND) if ET5_FOUND else None, "family": "T5", "max_length": "08번에서 지수 5번 규칙(Train 잘림 ≤0.1%)으로 결정", "input_prefix": "맞춤법 교정: ",
         "available_in_this_environment": bool(ET5_FOUND), "source": "동환 노트북"},
}
(P.CONFIG / "model_candidates.json").write_text(json.dumps(CAND, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", P.CONFIG / "model_candidates.json")
print({k: (v["model_id"] if isinstance(v, dict) else v) for k, v in CAND.items()})

저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/config/model_candidates.json
{'dataset_version': 'preprocessed_final_v1', 'KoBART': 'gogamza/kobart-base-v2', 'pko-T5': 'paust/pko-t5-base', 'ET5': None}


## 해석
- 후보 모델은 지수 프로젝트에서 실제 결과가 있는 **KoBART(BART 계열)**와 **pko-T5(T5 계열)**이다. 둘 다 Encoder–Decoder이며, 이 과제(입력 문장 → 교정 문장 생성)에 구조가 맞는다.
- ET5 가중치는 이 폴더 구조에서 **발견되지 않아 확인 불가**다. 따라서 08번의 ET5 실험은 기본적으로 꺼져 있고(RUN_ET5=False), ET5의 성능·학습시간에 대한 어떤 결론도 이 노트북 범위에서는 내리지 않는다.
- 동환 보고서에 적힌 ET5 구조 수치(24층·약 3.24억 파라미터·vocab 45,100)는 **이 환경에서 재확인하지 못한 값**이므로 출처 표기와 함께 참고용으로만 사용한다.
- 직접 결정한 것(모델 선택, max_length, 접두어, EOS 처리, 학습 설정)과 사전학습 모델에 이미 들어 있는 것(가중치, tokenizer, 구조)은 위 4절에서 구분했다. 라이브러리 모델을 불러온 것을 모델 설계라고 주장하지 않는다.